# LangChain 기반 KBO 모닝 브리핑

KBO 공식 홈페이지에서 전날 경기 기록을 가져오고, LangChain과 OpenAI 모델을 이용해 아침 브리핑을 생성하는 실습입니다.

- 외부 API: KBO 공식 홈페이지 경기 데이터
- AI 프레임워크: LangChain
- 생성 결과: 경기 결과를 근거로 작성한 구조화된 한국어 브리핑

## 1. 실습 목표

1. 외부 API에서 KBO 경기 데이터를 수집합니다.
2. 최종 점수, 승리·패전·세이브 투수, 결승타를 정리합니다.
3. `ChatPromptTemplate`과 `init_chat_model`을 사용합니다.
4. LCEL의 `prompt | model` 체인과 구조화 출력을 적용합니다.
5. AI가 입력에 없는 정보를 추측하지 않도록 프롬프트를 설계합니다.

## 2. 패키지 설치

In [1]:
%pip install -q --disable-pip-version-check httpx pandas pydantic langchain langchain-openai

Note: you may need to restart the kernel to use updated packages.


## 3. OpenAI API 키 설정

API 키는 노트북에 직접 작성하지 않습니다. 환경변수가 없다면 실행할 때 숨김 입력으로 받습니다. 입력한 값은 노트북 파일과 출력에 저장되지 않습니다.

In [2]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key를 입력하세요: ")

print("OPENAI_API_KEY 설정 완료")

OPENAI_API_KEY 설정 완료


## 4. 라이브러리와 상수 준비

재현 가능한 제출 결과를 위해 경기 날짜를 `2026-09-20`으로 지정했습니다. 실제 서비스에서는 한국 시간 기준 어제 날짜를 사용합니다.

In [3]:
import json
from datetime import date
from typing import Any

import httpx
import pandas as pd
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

KBO_GAME_LIST_URL = "https://www.koreabaseball.com/ws/Main.asmx/GetKboGameList"
KBO_BOX_SCORE_URL = "https://www.koreabaseball.com/ws/Schedule.asmx/GetBoxScoreScroll"
KBO_HEADERS = {
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "X-Requested-With": "XMLHttpRequest",
    "Referer": "https://www.koreabaseball.com/Schedule/GameCenter/Main.aspx",
}
KBO_SERIES_IDS = "0,1,3,4,5,6,7,8,9"
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
TARGET_DATE = date(2026, 9, 20)

KBO_TEAMS = {
    "HT": "KIA 타이거즈", "SS": "삼성 라이온즈",
    "LG": "LG 트윈스", "OB": "두산 베어스",
    "KT": "KT 위즈", "SK": "SSG 랜더스",
    "LT": "롯데 자이언츠", "NC": "NC 다이노스",
    "WO": "키움 히어로즈", "HH": "한화 이글스",
}

print(f"모델: {MODEL_NAME} / 조회 날짜: {TARGET_DATE}")

모델: gpt-4o-mini / 조회 날짜: 2026-09-20


## 5. KBO 경기 목록 가져오기

경기 목록 응답에서 팀, 점수, 경기 상태, 승리·패전·세이브 투수를 가져옵니다.

In [4]:
def clean_text(value: Any) -> str | None:
    if value is None:
        return None
    text = str(value).replace("&nbsp;", " " ).strip()
    return text or None


def parse_score(value: Any) -> int | None:
    text = clean_text(value)
    if text is None:
        return None
    try:
        return int(text)
    except ValueError:
        return None


def team_name(team_code: Any, fallback_name: Any) -> str:
    code = clean_text(team_code) or ""
    return KBO_TEAMS.get(code, clean_text(fallback_name) or code)


def fetch_kbo_game_list(target_date: date) -> list[dict[str, Any]]:
    form_data = {
        "leId": "1",
        "srId": KBO_SERIES_IDS,
        "date": target_date.strftime("%Y%m%d"),
    }
    response = httpx.post(
        KBO_GAME_LIST_URL,
        data=form_data,
        headers=KBO_HEADERS,
        timeout=10.0,
    )
    response.raise_for_status()
    payload = response.json()
    if str(payload.get("code")) != "100":
        raise RuntimeError(payload.get("msg", "KBO 경기 목록 조회 실패"))
    return payload.get("game") or []


raw_games = fetch_kbo_game_list(TARGET_DATE)
print(f"KBO 경기 목록 {len(raw_games)}경기 조회 완료")

KBO 경기 목록 5경기 조회 완료


## 6. 상세 기록에서 결승타 추출

완료된 각 경기의 박스스코어를 조회하고 `결승타` 행을 찾아냅니다. AI가 선수를 추측하지 않고 공식 기록만 사용하도록 입력 데이터를 먼저 확정합니다.

In [5]:
class GameResult(BaseModel):
    game_id: str
    home_team: str
    away_team: str
    home_score: int
    away_score: int
    winner: str | None = None
    loser: str | None = None
    winning_pitcher: str | None = None
    losing_pitcher: str | None = None
    save_pitcher: str | None = None
    winning_hit: str | None = None


class BriefingResult(BaseModel):
    headline: str = Field(description="전날 경기 결과를 표현하는 짧은 제목")
    summary: str = Field(description="공식 기록만 사용한 3~5문장의 한국어 요약")

In [6]:
def fetch_winning_hit(
    client: httpx.Client,
    game: dict[str, Any],
    target_date: date,
) -> str | None:
    form_data = {
        "leId": "1",
        "srId": str(game.get("SR_ID", 0)),
        "seasonId": str(game.get("SEASON_ID", target_date.year)),
        "gameId": game["G_ID"],
    }
    response = client.post(
        KBO_BOX_SCORE_URL, data=form_data, headers=KBO_HEADERS
    )
    response.raise_for_status()
    table_etc = json.loads(response.json().get("tableEtc") or "{}")

    for row_wrapper in table_etc.get("rows", []):
        cells = [
            clean_text(cell.get("Text"))
            for cell in row_wrapper.get("row", [])
        ]
        if len(cells) >= 2 and cells[0] == "결승타":
            return cells[1]
    return None


def build_game_results(
    games: list[dict[str, Any]], target_date: date
) -> list[GameResult]:
    results = []
    with httpx.Client(timeout=10.0) as client:
        for game in games:
            if game.get("GAME_RESULT_CK") != 1:
                continue

            home_score = parse_score(game.get("B_SCORE_CN"))
            away_score = parse_score(game.get("T_SCORE_CN"))
            if home_score is None or away_score is None:
                continue

            home_team = team_name(game.get("HOME_ID"), game.get("HOME_NM"))
            away_team = team_name(game.get("AWAY_ID"), game.get("AWAY_NM"))
            if home_score > away_score:
                winner, loser = home_team, away_team
            elif away_score > home_score:
                winner, loser = away_team, home_team
            else:
                winner, loser = None, None

            results.append(
                GameResult(
                    game_id=game["G_ID"],
                    home_team=home_team,
                    away_team=away_team,
                    home_score=home_score,
                    away_score=away_score,
                    winner=winner,
                    loser=loser,
                    winning_pitcher=clean_text(game.get("W_PIT_P_NM")),
                    losing_pitcher=clean_text(game.get("L_PIT_P_NM")),
                    save_pitcher=clean_text(game.get("SV_PIT_P_NM")),
                    winning_hit=fetch_winning_hit(client, game, target_date),
                )
            )
    return results


game_results = build_game_results(raw_games, TARGET_DATE)
print(f"상세 기록 {len(game_results)}경기 정리 완료")

상세 기록 5경기 정리 완료


### AI에 전달할 사실 데이터 확인

In [7]:
table_rows = [
    {
        "원정팀": game.away_team,
        "원정 점수": game.away_score,
        "홈 점수": game.home_score,
        "홈팀": game.home_team,
        "승리팀": game.winner,
        "패배팀": game.loser,
        "승리투수": game.winning_pitcher,
        "패전투수": game.losing_pitcher,
        "세이브": game.save_pitcher,
        "결승타": game.winning_hit,
    }
    for game in game_results
]
pd.DataFrame(table_rows)

,원정팀,원정 점수,홈 점수,홈팀,승리팀,패배팀,승리투수,패전투수,세이브,결승타
0,한화 이글스,3,4,LG 트윈스,LG 트윈스,한화 이글스,카라스코,황준서,손주영,문정빈(5회 2사 2루서 중전 안타)
1,삼성 라이온즈,6,13,롯데 자이언츠,롯데 자이언츠,삼성 라이온즈,이진하,원태인,NaN,레이예스(4회 1사 만루서 좌중간 안타)
2,키움 히어로즈,5,10,SSG 랜더스,SSG 랜더스,키움 히어로즈,김건우,하영민,NaN,에레디아(5회 1사 1루서 우중간 2루타)
3,두산 베어스,8,9,KT 위즈,KT 위즈,두산 베어스,고영표,박신지,김정운,"힐리어드(1회 1사 1,3루서 중견수 희생플라이)"
4,KIA 타이거즈,8,6,NC 다이노스,KIA 타이거즈,NC 다이노스,조상우,이용준,이의리,박정우(8회 2사 만루서 우익수 3루타)


## 7. LangChain 프롬프트 구성

시스템 프롬프트에서 제공된 데이터만 사용하도록 제한합니다. 선수 정보가 없는 경우에는 언급하지 않도록 지시하여 환각을 줄입니다.

In [8]:
SYSTEM_PROMPT = """
당신은 KBO 경기 결과를 전달하는 아침 브리핑 작성자입니다.
반드시 제공된 팀 이름, 최종 점수, 승리·패전·세이브 투수, 결승타 기록만 사용하세요.
winner는 승리팀, loser는 패배팀입니다. winning_pitcher는 항상 승리팀의 승리투수이고 losing_pitcher는 항상 패배팀의 패전투수입니다.
승리투수와 패전투수의 역할을 절대로 서로 바꾸어 표현하지 마세요.
입력에 없는 선수, 경기 장면, 순위, 기록은 추측하지 마세요.
값이 null이거나 비어 있는 항목은 언급하지 마세요.
과장된 표현이나 특정 팀을 비하하는 표현을 사용하지 마세요.
""".strip()

USER_PROMPT = """
[경기 날짜]
{game_date}

[공식 경기 결과 JSON]
{game_results}

위 기록을 바탕으로 짧은 제목과 3~5문장의 한국어 모닝 브리핑을 작성하세요.
""".strip()

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", USER_PROMPT),
])

print("ChatPromptTemplate 생성 완료")

ChatPromptTemplate 생성 완료


## 8. 모델과 LCEL 체인 생성

`with_structured_output()`으로 응답 형식을 고정하고, `prompt | structured_model` 형태로 LangChain 체인을 만듭니다.

In [9]:
model = init_chat_model(
    MODEL_NAME,
    model_provider="openai",
    temperature=0.2,
)
structured_model = model.with_structured_output(BriefingResult)
chain = prompt | structured_model

print("LCEL 체인 생성 완료: prompt | structured_model")

LCEL 체인 생성 완료: prompt | structured_model


## 9. AI 모닝 브리핑 생성

In [10]:
briefing = chain.invoke({
    "game_date": TARGET_DATE.isoformat(),
    "game_results": json.dumps(
        [game.model_dump() for game in game_results],
        ensure_ascii=False,
    ),
})

briefing

BriefingResult(headline='2026년 9월 20일 KBO 경기 결과', summary='9월 20일 KBO 리그에서 LG 트윈스가 한화 이글스를 4-3으로 이기며 승리했습니다. 승리투수는 카라스코, 패전투수는 황준서입니다. 롯데 자이언츠는 삼성 라이온즈를 13-6으로 제압하며 이진하가 승리투수로 활약했습니다. SSG 랜더스는 키움 히어로즈를 10-5로 이기고 김건우가 승리투수가 되었습니다. KT 위즈는 두산 베어스를 9-8로 이기며 고영표가 승리투수로 기록되었습니다. 마지막으로 KIA 타이거즈는 NC 다이노스를 8-6으로 이기며 조상우가 승리투수가 되었습니다.')

In [11]:
from IPython.display import Markdown, display

display(Markdown(
    f"## {briefing.headline}\n\n"
    f"{briefing.summary}\n\n"
    f"*데이터 기준: KBO 공식 홈페이지 · {TARGET_DATE}*"
))

## 2026년 9월 20일 KBO 경기 결과

9월 20일 KBO 리그에서 LG 트윈스가 한화 이글스를 4-3으로 이기며 승리했습니다. 승리투수는 카라스코, 패전투수는 황준서입니다. 롯데 자이언츠는 삼성 라이온즈를 13-6으로 제압하며 이진하가 승리투수로 활약했습니다. SSG 랜더스는 키움 히어로즈를 10-5로 이기고 김건우가 승리투수가 되었습니다. KT 위즈는 두산 베어스를 9-8로 이기며 고영표가 승리투수로 기록되었습니다. 마지막으로 KIA 타이거즈는 NC 다이노스를 8-6으로 이기며 조상우가 승리투수가 되었습니다.

*데이터 기준: KBO 공식 홈페이지 · 2026-09-20*

## 10. 웹 프로젝트 적용

노트북에서 검증한 흐름을 Vue 프로젝트에는 다음과 같이 적용했습니다.

```text
KBO 공식 경기 데이터
        ↓
FastAPI 백엔드에서 데이터 정리
        ↓
LangChain + OpenAI로 모닝 브리핑 생성
        ↓
Vue 화면에 오늘 경기와 전날 브리핑 표시
```

- `GET /api/games/today`: 오늘의 실제 KBO 경기 조회
- `POST /api/briefings/yesterday`: 전날 기록을 이용한 AI 브리핑 생성

이처럼 외부 스포츠 데이터와 생성형 AI를 결합해 기존 야구장 날씨 프로젝트를 확장했습니다.